# 04 — Entraînement et évaluation des modèles de régression

## Objectif

Ce notebook est consacré à l'entraînement, à la comparaison, à l'optimisation
et à l'évaluation des modèles de régression destinés à prédire les émissions
de CO₂ WLTP.

La variable cible est :

`co2_wltp_g_km`

Les données proviennent du preprocessing validé dans le notebook précédent :

`03_train_test_preprocessing.ipynb`

Les jeux suivants sont chargés depuis `data/processed/` :

- `X_train_processed.parquet` ;
- `X_test_processed.parquet` ;
- `y_train.parquet` ;
- `y_test.parquet`.

Le preprocessing amont fournit directement les variables numériques prêtes
pour la modélisation. La variable métier `manufacturer_make` est exclue du
modèle et ne doit pas réapparaître dans les matrices utilisées ici.

La méthodologie repose sur les principes suivants :

- le jeu de test final reste isolé pendant l'entraînement, la comparaison et
  l'optimisation des hyperparamètres ;
- les premières comparaisons sont réalisées sur un sous-échantillon de
  développement issu exclusivement de Train FULL ;
- ce sous-échantillon est séparé en Train développement et Validation
  développement ;
- les résultats numériques sont produits dynamiquement par les cellules de
  code et ne sont pas figés dans les cellules Markdown ;
- la sélection finale tient compte de la performance, de la généralisation et
  du coût de calcul ;
- le Test final n'est utilisé qu'après la sélection de la configuration
  candidate.

Les principales métriques sont :

- **MAE** — Mean Absolute Error ;
- **RMSE** — Root Mean Squared Error ;
- **R²** — coefficient de détermination.


## 1. Chargement des données prétraitées

### Objectif

Cette étape charge les jeux Train et Test produits par le pipeline de
preprocessing.

Aucune nouvelle transformation des variables n'est réalisée dans ce notebook.

Les contrôles portent sur :

- l'existence des fichiers ;
- la cohérence entre `X` et `y` ;
- l'identité de la structure des variables entre Train et Test ;
- l'absence de la cible dans les variables explicatives ;
- l'absence de `manufacturer_make` dans les variables de modélisation.


In [3]:
# =====================================================================
# 1 - Chargement des données prétraitées
# =====================================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# 1. Configuration générale
# ---------------------------------------------------------------------

TARGET_COLUMN = "co2_wltp_g_km"
RANDOM_STATE = 42


# ---------------------------------------------------------------------
# 2. Détermination robuste de la racine du projet
# ---------------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.parent != project_root
    and not (project_root / "pyproject.toml").exists()
):
    project_root = project_root.parent

if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError(
        "Impossible de déterminer la racine du projet : "
        "fichier pyproject.toml introuvable."
    )


# ---------------------------------------------------------------------
# 3. Chemins des datasets prétraités
# ---------------------------------------------------------------------

processed_data_dir = project_root / "data" / "processed"

x_train_path = processed_data_dir / "X_train_processed.parquet"
x_test_path = processed_data_dir / "X_test_processed.parquet"
y_train_path = processed_data_dir / "y_train.parquet"
y_test_path = processed_data_dir / "y_test.parquet"

required_files = {
    "X_train_processed": x_train_path,
    "X_test_processed": x_test_path,
    "y_train": y_train_path,
    "y_test": y_test_path,
}

missing_files = [
    name
    for name, path in required_files.items()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Fichiers prétraités manquants : " + ", ".join(missing_files)
    )


# ---------------------------------------------------------------------
# 4. Chargement
# ---------------------------------------------------------------------

print("=" * 72)
print("CHARGEMENT DES DONNÉES PRÉTRAITÉES")
print("=" * 72)

print(f"\nRacine du projet : {project_root}")
print(f"Répertoire source : {processed_data_dir}")

X_train = pd.read_parquet(x_train_path)
X_test = pd.read_parquet(x_test_path)
y_train = pd.read_parquet(y_train_path).squeeze("columns")
y_test = pd.read_parquet(y_test_path).squeeze("columns")


# ---------------------------------------------------------------------
# 5. Contrôles de cohérence
# ---------------------------------------------------------------------

if len(X_train) != len(y_train):
    raise ValueError("Incohérence entre X_train et y_train.")

if len(X_test) != len(y_test):
    raise ValueError("Incohérence entre X_test et y_test.")

if X_train.columns.tolist() != X_test.columns.tolist():
    raise ValueError(
        "X_train et X_test ne possèdent pas la même structure de variables."
    )

if TARGET_COLUMN in X_train.columns or TARGET_COLUMN in X_test.columns:
    raise ValueError(
        f"La cible '{TARGET_COLUMN}' est présente dans les variables explicatives."
    )

manufacturer_columns = [
    column
    for column in X_train.columns
    if "manufacturer_make" in column.lower()
]

if manufacturer_columns:
    raise ValueError(
        "La variable manufacturer_make ou une variable dérivée est encore "
        "présente dans les données de modélisation : "
        + ", ".join(manufacturer_columns)
    )


# ---------------------------------------------------------------------
# 6. Rapport dynamique
# ---------------------------------------------------------------------

datasets_report_df = pd.DataFrame(
    [
        {
            "dataset": "X_train",
            "observations": X_train.shape[0],
            "variables": X_train.shape[1],
            "rôle": "Train FULL disponible",
        },
        {
            "dataset": "X_test",
            "observations": X_test.shape[0],
            "variables": X_test.shape[1],
            "rôle": "Test final réservé",
        },
        {
            "dataset": "y_train",
            "observations": len(y_train),
            "variables": 1,
            "rôle": "Cible Train FULL",
        },
        {
            "dataset": "y_test",
            "observations": len(y_test),
            "variables": 1,
            "rôle": "Cible Test final",
        },
    ]
)

display(datasets_report_df)

print("\n✅ Données prétraitées chargées et cohérentes.")
print(f"✅ Nombre de variables explicatives : {X_train.shape[1]}")
print("✅ manufacturer_make est absente des variables de modélisation.")
print("✅ Le Test final est chargé mais reste réservé à l'évaluation finale.")


,dataset,observations,variables,rôle
0,X_train,8606547,35,Train FULL disponible
1,X_test,2151637,35,Test final réservé
2,y_train,8606547,1,Cible Train FULL
3,y_test,2151637,1,Cible Test final



✅ Données prétraitées chargées et cohérentes.
✅ Nombre de variables explicatives : 35
✅ manufacturer_make est absente des variables de modélisation.
✅ Le Test final est chargé mais reste réservé à l'évaluation finale.


## 2. Définition de la méthodologie d'évaluation

### 2.1 Objectif

Cette section définit un protocole commun à tous les modèles de régression.

### 2.2 Métriques retenues

Trois métriques complémentaires sont utilisées :

- **MAE** : erreur absolue moyenne, exprimée en g CO₂/km ;
- **RMSE** : racine de l'erreur quadratique moyenne, plus sensible aux erreurs
  importantes ;
- **R²** : proportion de la variance de la cible expliquée par le modèle.

### 2.3 Principe d'évaluation

Pendant la phase de développement, chaque modèle est entraîné sur
**Train développement** puis évalué sur **Validation développement**.

Le Test final n'intervient pas dans cette comparaison.

### 2.4 Temps d'entraînement

Le temps d'entraînement est mesuré afin de comparer le coût computationnel des
modèles, particulièrement important sur un dataset de plusieurs millions
d'observations.

### 2.5 Principe de comparaison

Les modèles sont comparés sur exactement les mêmes observations. Les écarts
Train développement / Validation développement servent à analyser la capacité
de généralisation.

### 2.6 Fonction commune d'évaluation

La cellule suivante définit une fonction unique utilisée par les trois modèles
afin d'éviter la duplication de la logique de calcul des métriques.


In [4]:
# =====================================================================
# 2.6 - Fonction commune d'évaluation des modèles de régression
# =====================================================================

import time

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


def evaluate_regression_model(
    *,
    model_name: str,
    model,
    X_train_eval: pd.DataFrame,
    y_train_eval: pd.Series,
    X_validation_eval: pd.DataFrame,
    y_validation_eval: pd.Series,
) -> dict:
    """
    Entraîne un modèle de régression et calcule MAE, RMSE et R²
    sur Train développement et Validation développement.
    """

    if len(X_train_eval) != len(y_train_eval):
        raise ValueError(
            f"{model_name}: incohérence entre X_train_eval et y_train_eval."
        )

    if len(X_validation_eval) != len(y_validation_eval):
        raise ValueError(
            f"{model_name}: incohérence entre X_validation_eval "
            "et y_validation_eval."
        )

    if X_train_eval.columns.tolist() != X_validation_eval.columns.tolist():
        raise ValueError(
            f"{model_name}: structures Train et Validation différentes."
        )

    start_time = time.perf_counter()
    model.fit(X_train_eval, y_train_eval)
    training_time_seconds = time.perf_counter() - start_time

    y_train_pred = model.predict(X_train_eval)
    y_validation_pred = model.predict(X_validation_eval)

    result = {
        "model": model_name,
        "mae_train": float(mean_absolute_error(y_train_eval, y_train_pred)),
        "mae_validation": float(
            mean_absolute_error(y_validation_eval, y_validation_pred)
        ),
        "rmse_train": float(
            np.sqrt(mean_squared_error(y_train_eval, y_train_pred))
        ),
        "rmse_validation": float(
            np.sqrt(mean_squared_error(y_validation_eval, y_validation_pred))
        ),
        "r2_train": float(r2_score(y_train_eval, y_train_pred)),
        "r2_validation": float(
            r2_score(y_validation_eval, y_validation_pred)
        ),
        "training_time_seconds": float(training_time_seconds),
    }

    return result


print("=" * 72)
print("FONCTION COMMUNE D'ÉVALUATION")
print("=" * 72)
print("\n✅ Fonction evaluate_regression_model définie.")
print("✅ Métriques : MAE, RMSE et R².")
print("✅ Mesure du temps d'entraînement intégrée.")
print("✅ Aucun modèle n'est entraîné dans cette cellule.")


FONCTION COMMUNE D'ÉVALUATION

✅ Fonction evaluate_regression_model définie.
✅ Métriques : MAE, RMSE et R².
✅ Mesure du temps d'entraînement intégrée.
✅ Aucun modèle n'est entraîné dans cette cellule.


## 3. Préparation des jeux de développement

### Objectif

Le jeu `X_train / y_train` constitue le Train FULL issu du preprocessing.
Afin de maîtriser les ressources pendant l'expérimentation, un
sous-échantillon reproductible est extrait exclusivement de Train FULL.

Ce sous-échantillon est ensuite séparé en :

- **Train développement** : entraînement des modèles ;
- **Validation développement** : comparaison et sélection des modèles.

Le Test final reste totalement isolé.

Si Train FULL contient moins d'observations que la taille de développement
prévue, la cellule utilise automatiquement toutes les observations disponibles
sans dépasser la taille du dataset.


In [5]:
# =====================================================================
# 3 - Préparation des jeux de développement
# =====================================================================

from sklearn.model_selection import train_test_split


# ---------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------

DEVELOPMENT_SAMPLE_SIZE = 500_000
VALIDATION_SIZE = 0.20


# ---------------------------------------------------------------------
# 2. Présentation de l'étape
# ---------------------------------------------------------------------

print("=" * 72)
print("PRÉPARATION DES JEUX DE DÉVELOPPEMENT")
print("=" * 72)

print(
    "\nContexte :"
    "\nLe preprocessing précédent a produit deux jeux indépendants :"
    f"\n  - Train FULL : {len(X_train):,} observations"
    f"\n  - Test final : {len(X_test):,} observations"
)


# ---------------------------------------------------------------------
# 3. Taille de développement réellement utilisable
# ---------------------------------------------------------------------

development_sample_size = min(
    DEVELOPMENT_SAMPLE_SIZE,
    len(X_train),
)

if development_sample_size < 2:
    raise ValueError(
        "Le Train FULL ne contient pas assez d'observations "
        "pour constituer les jeux de développement."
    )


# ---------------------------------------------------------------------
# 4. Sous-échantillon reproductible extrait exclusivement de Train FULL
# ---------------------------------------------------------------------

development_indices = X_train.sample(
    n=development_sample_size,
    random_state=RANDOM_STATE,
).index

X_development = X_train.loc[development_indices].copy()
y_development = y_train.loc[development_indices].copy()


# ---------------------------------------------------------------------
# 5. Séparation Train développement / Validation développement
# ---------------------------------------------------------------------

(
    X_train_dev,
    X_validation_dev,
    y_train_dev,
    y_validation_dev,
) = train_test_split(
    X_development,
    y_development,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)


# ---------------------------------------------------------------------
# 6. Contrôles de cohérence
# ---------------------------------------------------------------------

if len(X_train_dev) != len(y_train_dev):
    raise ValueError("Incohérence entre X_train_dev et y_train_dev.")

if len(X_validation_dev) != len(y_validation_dev):
    raise ValueError(
        "Incohérence entre X_validation_dev et y_validation_dev."
    )

if X_train_dev.columns.tolist() != X_validation_dev.columns.tolist():
    raise ValueError(
        "Train développement et Validation développement "
        "ne possèdent pas les mêmes variables."
    )

if not X_train_dev.index.intersection(X_validation_dev.index).empty:
    raise ValueError(
        "Fuite détectée : des observations sont communes à Train "
        "développement et Validation développement."
    )


# ---------------------------------------------------------------------
# 7. Rapport dynamique
# ---------------------------------------------------------------------

development_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train FULL",
            "observations": len(X_train),
            "variables": X_train.shape[1],
            "rôle": "Source du développement",
        },
        {
            "dataset": "Développement total",
            "observations": len(X_development),
            "variables": X_development.shape[1],
            "rôle": "Sous-échantillon de Train FULL",
        },
        {
            "dataset": "Train développement",
            "observations": len(X_train_dev),
            "variables": X_train_dev.shape[1],
            "rôle": "Entraînement des modèles",
        },
        {
            "dataset": "Validation développement",
            "observations": len(X_validation_dev),
            "variables": X_validation_dev.shape[1],
            "rôle": "Comparaison / sélection des modèles",
        },
        {
            "dataset": "Test final - réservé",
            "observations": len(X_test),
            "variables": X_test.shape[1],
            "rôle": "Évaluation finale uniquement",
        },
    ]
)

print("\n" + "=" * 72)
print("RÉSULTAT DE LA CONSTRUCTION DES JEUX")
print("=" * 72)
display(development_report_df)

print(
    f"\nTrain développement      : {len(X_train_dev):,} observations"
    f"\nValidation développement : {len(X_validation_dev):,} observations"
    f"\nTest final réservé        : {len(X_test):,} observations"
)

print("\n✅ Les jeux de développement proviennent exclusivement de Train FULL.")
print("✅ Train et Validation développement ne partagent aucune observation.")
print("✅ X_test / y_test reste réservé à l'évaluation finale.")
print(f"✅ Séparation reproductible avec random_state = {RANDOM_STATE}.")


PRÉPARATION DES JEUX DE DÉVELOPPEMENT

Contexte :
Le preprocessing précédent a produit deux jeux indépendants :
  - Train FULL : 8,606,547 observations
  - Test final : 2,151,637 observations

RÉSULTAT DE LA CONSTRUCTION DES JEUX


,dataset,observations,variables,rôle
0,Train FULL,8606547,35,Source du développement
1,Développement total,500000,35,Sous-échantillon de Train FULL
2,Train développement,400000,35,Entraînement des modèles
3,Validation développement,100000,35,Comparaison / sélection des modèles
4,Test final - réservé,2151637,35,Évaluation finale uniquement



Train développement      : 400,000 observations
Validation développement : 100,000 observations
Test final réservé        : 2,151,637 observations

✅ Les jeux de développement proviennent exclusivement de Train FULL.
✅ Train et Validation développement ne partagent aucune observation.
✅ X_test / y_test reste réservé à l'évaluation finale.
✅ Séparation reproductible avec random_state = 42.


## 4. Modèle baseline — Régression Ridge

### 4.1 Objectif

La Régression Ridge constitue la baseline linéaire du notebook. Elle fournit
un niveau de performance de référence avant l'étude des modèles non linéaires.

### 4.2 Données utilisées

Le modèle est entraîné sur `X_train_dev / y_train_dev` et évalué sur
`X_validation_dev / y_validation_dev`.

Le Test final reste strictement isolé.

### 4.3 Métriques

MAE, RMSE, R² et temps d'entraînement sont calculés dynamiquement.


In [6]:
# =====================================================================
# 4 - Modèle baseline : Régression Ridge
# =====================================================================

from sklearn.linear_model import Ridge


print("=" * 72)
print("MODÈLE BASELINE — RÉGRESSION RIDGE")
print("=" * 72)

print(
    "\nDonnées utilisées :"
    f"\n  - Train développement      : {len(X_train_dev):,} observations"
    f"\n  - Validation développement : {len(X_validation_dev):,} observations"
    f"\n  - Nombre de variables      : {X_train_dev.shape[1]}"
)
print("\nLe jeu X_test / y_test n'est pas utilisé à cette étape.")

ridge_model = Ridge(alpha=1.0)

ridge_result = evaluate_regression_model(
    model_name="Ridge",
    model=ridge_model,
    X_train_eval=X_train_dev,
    y_train_eval=y_train_dev,
    X_validation_eval=X_validation_dev,
    y_validation_eval=y_validation_dev,
)

ridge_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train développement",
            "observations": len(X_train_dev),
            "MAE_g_co2_km": ridge_result["mae_train"],
            "RMSE_g_co2_km": ridge_result["rmse_train"],
            "R2": ridge_result["r2_train"],
        },
        {
            "dataset": "Validation développement",
            "observations": len(X_validation_dev),
            "MAE_g_co2_km": ridge_result["mae_validation"],
            "RMSE_g_co2_km": ridge_result["rmse_validation"],
            "R2": ridge_result["r2_validation"],
        },
    ]
)

mae_gap = ridge_result["mae_validation"] - ridge_result["mae_train"]
rmse_gap = ridge_result["rmse_validation"] - ridge_result["rmse_train"]
r2_gap = ridge_result["r2_train"] - ridge_result["r2_validation"]

print("\n" + "=" * 72)
print("RÉSULTATS RIDGE")
print("=" * 72)
display(ridge_report_df)

print(
    f"\nÉcart Train / Validation :"
    f"\n  - Δ MAE  : {mae_gap:+.4f} g CO₂/km"
    f"\n  - Δ RMSE : {rmse_gap:+.4f} g CO₂/km"
    f"\n  - Δ R²   : {r2_gap:+.4f}"
    f"\n\nTemps d'entraînement : {ridge_result['training_time_seconds']:.2f} secondes"
)

print("\n✅ Ridge entraîné exclusivement sur Train développement.")
print("✅ Généralisation mesurée sur Validation développement.")
print("✅ X_test / y_test reste totalement isolé.")


MODÈLE BASELINE — RÉGRESSION RIDGE

Données utilisées :
  - Train développement      : 400,000 observations
  - Validation développement : 100,000 observations
  - Nombre de variables      : 35

Le jeu X_test / y_test n'est pas utilisé à cette étape.

RÉSULTATS RIDGE


,dataset,observations,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Train développement,400000,6.746360,11.548451,0.960173
1,Validation développement,100000,6.701778,11.438299,0.960865



Écart Train / Validation :
  - Δ MAE  : -0.0446 g CO₂/km
  - Δ RMSE : -0.1102 g CO₂/km
  - Δ R²   : -0.0007

Temps d'entraînement : 0.47 secondes

✅ Ridge entraîné exclusivement sur Train développement.
✅ Généralisation mesurée sur Validation développement.
✅ X_test / y_test reste totalement isolé.


## 5. Random Forest Regressor

### 5.1 Objectif

Random Forest est le premier modèle non linéaire comparé à la baseline Ridge.
Il peut capturer des relations complexes et des interactions entre variables.

### 5.2 Données utilisées

Il utilise exactement les mêmes jeux Train développement / Validation
développement que Ridge afin de garantir une comparaison équitable.

### 5.3 Paramétrage initial

La configuration utilisée ici est une configuration de référence, avant la
phase d'optimisation des hyperparamètres.


In [7]:
# =====================================================================
# 5 - Random Forest Regressor
# =====================================================================

from sklearn.ensemble import RandomForestRegressor


RF_PARAMS = {
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

print("=" * 72)
print("MODÈLE NON LINÉAIRE — RANDOM FOREST REGRESSOR")
print("=" * 72)

print(
    "\nDonnées utilisées :"
    f"\n  - Train développement      : {len(X_train_dev):,} observations"
    f"\n  - Validation développement : {len(X_validation_dev):,} observations"
    f"\n  - Nombre de variables      : {X_train_dev.shape[1]}"
)
print("\nLe jeu X_test / y_test reste strictement isolé.")

print("\nParamètres Random Forest :")
for parameter, value in RF_PARAMS.items():
    print(f"  - {parameter} : {value}")

random_forest_model = RandomForestRegressor(**RF_PARAMS)

random_forest_result = evaluate_regression_model(
    model_name="Random Forest",
    model=random_forest_model,
    X_train_eval=X_train_dev,
    y_train_eval=y_train_dev,
    X_validation_eval=X_validation_dev,
    y_validation_eval=y_validation_dev,
)

random_forest_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train développement",
            "observations": len(X_train_dev),
            "MAE_g_co2_km": random_forest_result["mae_train"],
            "RMSE_g_co2_km": random_forest_result["rmse_train"],
            "R2": random_forest_result["r2_train"],
        },
        {
            "dataset": "Validation développement",
            "observations": len(X_validation_dev),
            "MAE_g_co2_km": random_forest_result["mae_validation"],
            "RMSE_g_co2_km": random_forest_result["rmse_validation"],
            "R2": random_forest_result["r2_validation"],
        },
    ]
)

rf_mae_gap = random_forest_result["mae_validation"] - random_forest_result["mae_train"]
rf_rmse_gap = random_forest_result["rmse_validation"] - random_forest_result["rmse_train"]
rf_r2_gap = random_forest_result["r2_train"] - random_forest_result["r2_validation"]

print("\n" + "=" * 72)
print("RÉSULTATS RANDOM FOREST")
print("=" * 72)
display(random_forest_report_df)

print(
    f"\nÉcart Train / Validation :"
    f"\n  - Δ MAE  : {rf_mae_gap:+.4f} g CO₂/km"
    f"\n  - Δ RMSE : {rf_rmse_gap:+.4f} g CO₂/km"
    f"\n  - Δ R²   : {rf_r2_gap:+.4f}"
    f"\n\nTemps d'entraînement : {random_forest_result['training_time_seconds']:.2f} secondes"
)

comparison_ridge_rf_df = pd.DataFrame(
    [
        {
            "modèle": "Ridge",
            "MAE_validation": ridge_result["mae_validation"],
            "RMSE_validation": ridge_result["rmse_validation"],
            "R2_validation": ridge_result["r2_validation"],
            "temps_entraînement_s": ridge_result["training_time_seconds"],
        },
        {
            "modèle": "Random Forest",
            "MAE_validation": random_forest_result["mae_validation"],
            "RMSE_validation": random_forest_result["rmse_validation"],
            "R2_validation": random_forest_result["r2_validation"],
            "temps_entraînement_s": random_forest_result["training_time_seconds"],
        },
    ]
)

print("\n" + "=" * 72)
print("COMPARAISON AVEC LA BASELINE RIDGE")
print("=" * 72)
display(comparison_ridge_rf_df)

print("\n✅ Ridge et Random Forest ont été comparés sur le même jeu de validation.")
print("✅ X_test / y_test reste totalement isolé.")


MODÈLE NON LINÉAIRE — RANDOM FOREST REGRESSOR

Données utilisées :
  - Train développement      : 400,000 observations
  - Validation développement : 100,000 observations
  - Nombre de variables      : 35

Le jeu X_test / y_test reste strictement isolé.

Paramètres Random Forest :
  - n_estimators : 100
  - max_depth : None
  - min_samples_split : 2
  - min_samples_leaf : 1
  - max_features : 1.0
  - n_jobs : -1
  - random_state : 42

RÉSULTATS RANDOM FOREST


,dataset,observations,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Train développement,400000,0.172750,1.233287,0.999546
1,Validation développement,100000,0.399097,2.861367,0.997551



Écart Train / Validation :
  - Δ MAE  : +0.2263 g CO₂/km
  - Δ RMSE : +1.6281 g CO₂/km
  - Δ R²   : +0.0020

Temps d'entraînement : 122.45 secondes

COMPARAISON AVEC LA BASELINE RIDGE


,modèle,MAE_validation,RMSE_validation,R2_validation,temps_entraînement_s
0,Ridge,6.701778,11.438299,0.960865,0.471225
1,Random Forest,0.399097,2.861367,0.997551,122.450519



✅ Ridge et Random Forest ont été comparés sur le même jeu de validation.
✅ X_test / y_test reste totalement isolé.


## 6. Contrôle anti-data-leakage des variables explicatives

### 6.1 Objectif

Avant de poursuivre, cette étape vérifie la structure des variables utilisées
par les modèles.

Le contrôle porte sur :

- la liste exhaustive des variables ;
- l'absence de la cible dans `X` ;
- l'absence de `manufacturer_make` et de variables dérivées de cette colonne ;
- l'identité des colonnes entre Train développement, Validation développement
  et Test final.

Cette étape ne réentraîne aucun modèle et n'utilise pas `y_test`.


In [8]:
# =====================================================================
# 6 - Contrôle anti-data-leakage des variables explicatives
# =====================================================================

print("=" * 72)
print("CONTRÔLE ANTI-DATA LEAKAGE — VARIABLES EXPLICATIVES")
print("=" * 72)

print(f"\nNombre de variables explicatives : {X_train_dev.shape[1]}")
print("\nVariables utilisées :\n")

for index, column in enumerate(X_train_dev.columns, start=1):
    print(f"{index:02d}. {column}")

forbidden_exact_columns = {TARGET_COLUMN}
forbidden_patterns = ("manufacturer_make",)

forbidden_columns = [
    column
    for column in X_train_dev.columns
    if column in forbidden_exact_columns
    or any(pattern in column.lower() for pattern in forbidden_patterns)
]

same_train_validation = (
    X_train_dev.columns.tolist() == X_validation_dev.columns.tolist()
)
same_train_test = X_train_dev.columns.tolist() == X_test.columns.tolist()

print("\n" + "=" * 72)
print("RÉSULTAT DU CONTRÔLE")
print("=" * 72)

if forbidden_columns:
    raise ValueError(
        "Variable(s) interdite(s) détectée(s) : " + ", ".join(forbidden_columns)
    )

if not same_train_validation:
    raise ValueError(
        "Les colonnes Train développement / Validation développement diffèrent."
    )

if not same_train_test:
    raise ValueError(
        "Les colonnes Train développement / Test final diffèrent."
    )

print("\n✅ La cible n'est pas présente dans les variables explicatives.")
print("✅ manufacturer_make est absente des variables de modélisation.")
print("✅ Train développement, Validation développement et Test final ont la même structure.")
print("✅ Aucun entraînement supplémentaire n'a été réalisé.")
print("✅ y_test n'a pas été utilisé.")


CONTRÔLE ANTI-DATA LEAKAGE — VARIABLES EXPLICATIVES

Nombre de variables explicatives : 35

Variables utilisées :

01. mass_running_order_kg
02. wltp_test_mass_kg
03. engine_capacity_cm3
04. engine_power_kw
05. electric_energy_consumption_wh_km
06. co2_reduction_wltp_g_km
07. fuel_consumption
08. electric_range_km
09. registration_month_sin
10. registration_month_cos
11. has_electric_energy_consumption_wh_km
12. has_electric_range_km
13. has_fuel_consumption
14. has_co2_reduction_wltp_g_km
15. vehicle_category_type_M1
16. vehicle_category_type_M1G
17. vehicle_category_type_N1
18. vehicle_category_type_N1G
19. vehicle_category_type_N2
20. fuel_type_diesel
21. fuel_type_diesel/electric
22. fuel_type_e85
23. fuel_type_electric
24. fuel_type_hydrogen
25. fuel_type_lpg
26. fuel_type_ng
27. fuel_type_petrol
28. fuel_type_petrol/electric
29. fuel_type_unknown
30. fuel_mode_B
31. fuel_mode_E
32. fuel_mode_F
33. fuel_mode_H
34. fuel_mode_M
35. fuel_mode_P

RÉSULTAT DU CONTRÔLE

✅ La cible n'est

## 7. HistGradientBoosting Regressor

### 7.1 Objectif

HistGradientBoosting est le troisième modèle évalué. Cette famille est adaptée
aux jeux volumineux et offre généralement un bon compromis entre performance
et coût de calcul.

Le modèle utilise exactement les mêmes jeux de développement que Ridge et
Random Forest.

Le Test final reste isolé.


In [9]:
# =====================================================================
# 7 - HistGradientBoosting Regressor
# =====================================================================

from sklearn.ensemble import HistGradientBoostingRegressor


HGB_PARAMS = {
    "learning_rate": 0.1,
    "max_iter": 200,
    "max_leaf_nodes": 31,
    "l2_regularization": 0.0,
    "random_state": RANDOM_STATE,
}

print("=" * 72)
print("MODÈLE NON LINÉAIRE — HISTGRADIENTBOOSTING REGRESSOR")
print("=" * 72)

print(
    "\nDonnées utilisées :"
    f"\n  - Train développement      : {len(X_train_dev):,} observations"
    f"\n  - Validation développement : {len(X_validation_dev):,} observations"
    f"\n  - Nombre de variables      : {X_train_dev.shape[1]}"
)
print("\nLe jeu X_test / y_test reste strictement isolé.")

print("\nParamètres HistGradientBoosting :")
for parameter, value in HGB_PARAMS.items():
    print(f"  - {parameter} : {value}")

hist_gradient_boosting_model = HistGradientBoostingRegressor(**HGB_PARAMS)

hist_gradient_boosting_result = evaluate_regression_model(
    model_name="HistGradientBoosting",
    model=hist_gradient_boosting_model,
    X_train_eval=X_train_dev,
    y_train_eval=y_train_dev,
    X_validation_eval=X_validation_dev,
    y_validation_eval=y_validation_dev,
)

hist_gradient_boosting_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train développement",
            "observations": len(X_train_dev),
            "MAE_g_co2_km": hist_gradient_boosting_result["mae_train"],
            "RMSE_g_co2_km": hist_gradient_boosting_result["rmse_train"],
            "R2": hist_gradient_boosting_result["r2_train"],
        },
        {
            "dataset": "Validation développement",
            "observations": len(X_validation_dev),
            "MAE_g_co2_km": hist_gradient_boosting_result["mae_validation"],
            "RMSE_g_co2_km": hist_gradient_boosting_result["rmse_validation"],
            "R2": hist_gradient_boosting_result["r2_validation"],
        },
    ]
)

hgb_mae_gap = hist_gradient_boosting_result["mae_validation"] - hist_gradient_boosting_result["mae_train"]
hgb_rmse_gap = hist_gradient_boosting_result["rmse_validation"] - hist_gradient_boosting_result["rmse_train"]
hgb_r2_gap = hist_gradient_boosting_result["r2_train"] - hist_gradient_boosting_result["r2_validation"]

print("\n" + "=" * 72)
print("RÉSULTATS HISTGRADIENTBOOSTING")
print("=" * 72)
display(hist_gradient_boosting_report_df)

print(
    f"\nÉcart Train / Validation :"
    f"\n  - Δ MAE  : {hgb_mae_gap:+.4f} g CO₂/km"
    f"\n  - Δ RMSE : {hgb_rmse_gap:+.4f} g CO₂/km"
    f"\n  - Δ R²   : {hgb_r2_gap:+.4f}"
    f"\n\nTemps d'entraînement : {hist_gradient_boosting_result['training_time_seconds']:.2f} secondes"
)

print("\n✅ HistGradientBoosting entraîné sur Train développement uniquement.")
print("✅ X_test / y_test reste totalement isolé.")


MODÈLE NON LINÉAIRE — HISTGRADIENTBOOSTING REGRESSOR

Données utilisées :
  - Train développement      : 400,000 observations
  - Validation développement : 100,000 observations
  - Nombre de variables      : 35

Le jeu X_test / y_test reste strictement isolé.

Paramètres HistGradientBoosting :
  - learning_rate : 0.1
  - max_iter : 200
  - max_leaf_nodes : 31
  - l2_regularization : 0.0
  - random_state : 42

RÉSULTATS HISTGRADIENTBOOSTING


,dataset,observations,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Train développement,400000,1.472714,3.762700,0.995772
1,Validation développement,100000,1.508407,4.038007,0.995123



Écart Train / Validation :
  - Δ MAE  : +0.0357 g CO₂/km
  - Δ RMSE : +0.2753 g CO₂/km
  - Δ R²   : +0.0006

Temps d'entraînement : 13.72 secondes

✅ HistGradientBoosting entraîné sur Train développement uniquement.
✅ X_test / y_test reste totalement isolé.


### 7.2 Comparaison consolidée des modèles de régression

Les trois modèles sont comparés sur exactement le même jeu Validation
développement.

Le tableau consolidé calcule également les écarts Train développement /
Validation développement afin d'analyser la généralisation.

Le **RMSE de validation** est utilisé comme critère principal de classement,
avec la MAE, le R² et le temps d'entraînement comme critères complémentaires.


In [ ]:
# =====================================================================
# 7.2 - Comparaison consolidée des modèles de régression
# =====================================================================

regression_comparison_df = pd.DataFrame(
    [
        ridge_result,
        random_forest_result,
        hist_gradient_boosting_result,
    ]
)

regression_comparison_df["mae_gap"] = (
    regression_comparison_df["mae_validation"]
    - regression_comparison_df["mae_train"]
)
regression_comparison_df["rmse_gap"] = (
    regression_comparison_df["rmse_validation"]
    - regression_comparison_df["rmse_train"]
)
regression_comparison_df["r2_gap"] = (
    regression_comparison_df["r2_train"]
    - regression_comparison_df["r2_validation"]
)

regression_comparison_df = regression_comparison_df[
    [
        "model",
        "mae_train",
        "mae_validation",
        "mae_gap",
        "rmse_train",
        "rmse_validation",
        "rmse_gap",
        "r2_train",
        "r2_validation",
        "r2_gap",
        "training_time_seconds",
    ]
].sort_values("rmse_validation").reset_index(drop=True)

comparison_display_df = regression_comparison_df.copy()
numeric_columns = comparison_display_df.select_dtypes(include="number").columns
comparison_display_df[numeric_columns] = comparison_display_df[numeric_columns].round(4)

print("=" * 72)
print("COMPARAISON CONSOLIDÉE DES MODÈLES")
print("=" * 72)
print("\nClassement principal : RMSE Validation développement croissant.\n")
display(comparison_display_df)

print("\n✅ Les trois modèles ont été évalués sur le même jeu de validation.")
print("✅ X_test / y_test n'intervient pas dans ce classement.")


COMPARAISON CONSOLIDÉE DES MODÈLES

Classement principal : RMSE Validation développement croissant.



,model,mae_train,mae_validation,mae_gap,rmse_train,rmse_validation,rmse_gap,r2_train,r2_validation,r2_gap,training_time_seconds
0,Random Forest,0.1727,0.3991,0.2263,1.2333,2.8614,1.6281,0.9995,0.9976,0.0020,122.4505
1,HistGradientBoosting,1.4727,1.5084,0.0357,3.7627,4.0380,0.2753,0.9958,0.9951,0.0006,13.7237
2,Ridge,6.7464,6.7018,-0.0446,11.5485,11.4383,-0.1102,0.9602,0.9609,-0.0007,0.4712



✅ Les trois modèles ont été évalués sur le même jeu de validation.
✅ X_test / y_test n'intervient pas dans ce classement.


### 7.3 Interprétation des résultats

La comparaison est interprétée selon trois dimensions :

- **précision prédictive** : MAE, RMSE et R² sur Validation développement ;
- **généralisation** : écarts Train développement / Validation développement ;
- **coût de calcul** : temps d'entraînement.

Aucun résultat numérique n'est inscrit en dur dans ce Markdown. La cellule de
code associée produit l'interprétation à partir de l'exécution courante.

Le Test final reste isolé.


In [ ]:
# =====================================================================
# 7.3 - Interprétation dynamique des résultats
# =====================================================================

best_mae_model = regression_comparison_df.loc[
    regression_comparison_df["mae_validation"].idxmin()
]
best_rmse_model = regression_comparison_df.loc[
    regression_comparison_df["rmse_validation"].idxmin()
]
best_r2_model = regression_comparison_df.loc[
    regression_comparison_df["r2_validation"].idxmax()
]
fastest_model = regression_comparison_df.loc[
    regression_comparison_df["training_time_seconds"].idxmin()
]
most_stable_rmse_model = regression_comparison_df.loc[
    regression_comparison_df["rmse_gap"].abs().idxmin()
]

print("=" * 72)
print("INTERPRÉTATION DYNAMIQUE DES RÉSULTATS")
print("=" * 72)

print(
    f"\nMeilleure MAE Validation : {best_mae_model['model']} "
    f"({best_mae_model['mae_validation']:.4f} g CO₂/km)"
)
print(
    f"Meilleur RMSE Validation : {best_rmse_model['model']} "
    f"({best_rmse_model['rmse_validation']:.4f} g CO₂/km)"
)
print(
    f"Meilleur R² Validation   : {best_r2_model['model']} "
    f"({best_r2_model['r2_validation']:.4f})"
)
print(
    f"Modèle le plus rapide    : {fastest_model['model']} "
    f"({fastest_model['training_time_seconds']:.2f} s)"
)
print(
    f"Plus faible écart RMSE   : {most_stable_rmse_model['model']} "
    f"({most_stable_rmse_model['rmse_gap']:+.4f} g CO₂/km)"
)

print("\n✅ Interprétation produite à partir des résultats de l'exécution courante.")
print("✅ X_test / y_test reste totalement isolé.")


INTERPRÉTATION DYNAMIQUE DES RÉSULTATS

Meilleure MAE Validation : Random Forest (0.3991 g CO₂/km)
Meilleur RMSE Validation : Random Forest (2.8614 g CO₂/km)
Meilleur R² Validation   : Random Forest (0.9976)
Modèle le plus rapide    : Ridge (0.47 s)
Plus faible écart RMSE   : Ridge (-0.1102 g CO₂/km)

✅ Interprétation produite à partir des résultats de l'exécution courante.
✅ X_test / y_test reste totalement isolé.


### 7.4 Analyse dynamique des performances

Cette étape détaille automatiquement les performances de chaque modèle sur
Train développement et Validation développement.

Elle permet de visualiser simultanément :

- les métriques de validation ;
- les écarts de généralisation ;
- le temps d'entraînement.

Les données chiffrées sont calculées dynamiquement et ne sont pas figées dans
le Markdown.


In [ ]:
# =====================================================================
# 7.4 - Analyse dynamique des performances
# =====================================================================

print("=" * 72)
print("ANALYSE DYNAMIQUE DES PERFORMANCES")
print("=" * 72)

for _, row in regression_comparison_df.sort_values("rmse_validation").iterrows():
    print(
        f"\n{row['model']}"
        f"\n  - MAE Train       : {row['mae_train']:.4f} g CO₂/km"
        f"\n  - MAE Validation  : {row['mae_validation']:.4f} g CO₂/km"
        f"\n  - Écart MAE       : {row['mae_gap']:+.4f} g CO₂/km"
        f"\n  - RMSE Train      : {row['rmse_train']:.4f} g CO₂/km"
        f"\n  - RMSE Validation : {row['rmse_validation']:.4f} g CO₂/km"
        f"\n  - Écart RMSE      : {row['rmse_gap']:+.4f} g CO₂/km"
        f"\n  - R² Train        : {row['r2_train']:.4f}"
        f"\n  - R² Validation   : {row['r2_validation']:.4f}"
        f"\n  - Écart R²        : {row['r2_gap']:+.4f}"
        f"\n  - Temps Train     : {row['training_time_seconds']:.2f} secondes"
    )

print("\n" + "=" * 72)
print("CONCLUSION DYNAMIQUE")
print("=" * 72)

if (
    best_mae_model["model"]
    == best_rmse_model["model"]
    == best_r2_model["model"]
):
    print(
        f"\n✅ {best_rmse_model['model']} présente les meilleures performances "
        "de validation sur MAE, RMSE et R²."
    )
else:
    print("\nLes métriques ne désignent pas toutes le même meilleur modèle.")

print(f"Le modèle le plus rapide est {fastest_model['model']}.")
print(
    f"Le modèle le plus stable selon l'écart RMSE est "
    f"{most_stable_rmse_model['model']}."
)
print("✅ X_test / y_test reste totalement isolé.")


ANALYSE DYNAMIQUE DES PERFORMANCES

Random Forest
  - MAE Train       : 0.1727 g CO₂/km
  - MAE Validation  : 0.3991 g CO₂/km
  - Écart MAE       : +0.2263 g CO₂/km
  - RMSE Train      : 1.2333 g CO₂/km
  - RMSE Validation : 2.8614 g CO₂/km
  - Écart RMSE      : +1.6281 g CO₂/km
  - R² Train        : 0.9995
  - R² Validation   : 0.9976
  - Écart R²        : +0.0020
  - Temps Train     : 122.45 secondes

HistGradientBoosting
  - MAE Train       : 1.4727 g CO₂/km
  - MAE Validation  : 1.5084 g CO₂/km
  - Écart MAE       : +0.0357 g CO₂/km
  - RMSE Train      : 3.7627 g CO₂/km
  - RMSE Validation : 4.0380 g CO₂/km
  - Écart RMSE      : +0.2753 g CO₂/km
  - R² Train        : 0.9958
  - R² Validation   : 0.9951
  - Écart R²        : +0.0006
  - Temps Train     : 13.72 secondes

Ridge
  - MAE Train       : 6.7464 g CO₂/km
  - MAE Validation  : 6.7018 g CO₂/km
  - Écart MAE       : -0.0446 g CO₂/km
  - RMSE Train      : 11.5485 g CO₂/km
  - RMSE Validation : 11.4383 g CO₂/km
  - Écart RMSE   

### 7.5 Conclusion de la comparaison et sélection des modèles candidats

La phase suivante optimise les modèles non linéaires retenus.

La sélection des candidats repose sur :

- leurs performances sur Validation développement ;
- leur capacité de généralisation ;
- leur coût d'entraînement.

Ridge conserve son rôle de baseline. Random Forest et
HistGradientBoosting restent les deux candidats de la phase d'optimisation.

Aucun modèle final n'est sélectionné à ce stade.


In [ ]:
# =====================================================================
# 7.5 - Conclusion et sélection dynamique des modèles candidats
# =====================================================================

baseline_model = regression_comparison_df.loc[
    regression_comparison_df["model"] == "Ridge"
].iloc[0]

non_linear_models = regression_comparison_df.loc[
    regression_comparison_df["model"] != "Ridge"
].copy()

if len(non_linear_models) != 2:
    raise ValueError(
        "Deux modèles non linéaires sont attendus : "
        "Random Forest et HistGradientBoosting."
    )

performance_candidate = non_linear_models.loc[
    non_linear_models["rmse_validation"].idxmin()
]

non_linear_models["rank_performance"] = (
    non_linear_models["rmse_validation"].rank(method="min")
)
non_linear_models["rank_generalization"] = (
    non_linear_models["rmse_gap"].abs().rank(method="min")
)
non_linear_models["rank_training_time"] = (
    non_linear_models["training_time_seconds"].rank(method="min")
)
non_linear_models["compromise_score"] = non_linear_models[
    ["rank_performance", "rank_generalization", "rank_training_time"]
].mean(axis=1)

compromise_candidate = non_linear_models.loc[
    non_linear_models["compromise_score"].idxmin()
]

selected_candidates = ["Random Forest", "HistGradientBoosting"]

print("=" * 72)
print("CONCLUSION ET SÉLECTION DES MODÈLES CANDIDATS")
print("=" * 72)

print(f"\nBaseline : {baseline_model['model']}")
print(
    f"Candidat orienté performance : {performance_candidate['model']} "
    f"(RMSE Validation={performance_candidate['rmse_validation']:.4f})"
)
print(
    f"Candidat orienté compromis   : {compromise_candidate['model']} "
    f"(score={compromise_candidate['compromise_score']:.2f})"
)

print("\nModèles retenus pour l'optimisation :")
for index, model_name in enumerate(selected_candidates, start=1):
    print(f"  {index}. {model_name}")

print("\n✅ Ridge reste la baseline de référence.")
print("✅ Les deux modèles non linéaires passent à l'optimisation.")
print("✅ Aucun modèle final n'est sélectionné à ce stade.")
print("✅ X_test / y_test reste totalement isolé.")


CONCLUSION ET SÉLECTION DES MODÈLES CANDIDATS

Baseline : Ridge
Candidat orienté performance : Random Forest (RMSE Validation=2.8614)
Candidat orienté compromis   : HistGradientBoosting (score=1.33)

Modèles retenus pour l'optimisation :
  1. Random Forest
  2. HistGradientBoosting

✅ Ridge reste la baseline de référence.
✅ Les deux modèles non linéaires passent à l'optimisation.
✅ Aucun modèle final n'est sélectionné à ce stade.
✅ X_test / y_test reste totalement isolé.


## 8. Optimisation des hyperparamètres

### Objectif

Les configurations précédentes sont des configurations initiales. L'objectif
est maintenant d'optimiser :

- **Random Forest Regressor** ;
- **HistGradientBoosting Regressor**.

Compte tenu du volume des données, la stratégie d'optimisation est progressive
et contrôlée.

Le Test final reste exclu de la recherche, de la comparaison des configurations
et de la sélection du meilleur candidat.


### 8.1 Constitution des données pour l'optimisation des hyperparamètres

Les jeux nécessaires au tuning existent déjà :

- `X_train_dev / y_train_dev` pour l'apprentissage des configurations ;
- `X_validation_dev / y_validation_dev` pour la validation externe des
  meilleures configurations.

Aucun nouveau découpage n'est effectué. Cette réutilisation garantit que les
configurations optimisées sont comparées sur le même jeu de validation que les
configurations initiales.

Le Test final reste totalement exclu du tuning.


In [14]:
# =====================================================================
# 8.1 - Constitution des données pour l'optimisation
# =====================================================================

print("=" * 72)
print("PRÉPARATION DES DONNÉES POUR L'OPTIMISATION")
print("=" * 72)

X_train_tuning = X_train_dev
y_train_tuning = y_train_dev
X_validation_tuning = X_validation_dev
y_validation_tuning = y_validation_dev

if len(X_train_tuning) != len(y_train_tuning):
    raise ValueError("Incohérence entre X_train_tuning et y_train_tuning.")

if len(X_validation_tuning) != len(y_validation_tuning):
    raise ValueError(
        "Incohérence entre X_validation_tuning et y_validation_tuning."
    )

if X_train_tuning.columns.tolist() != X_validation_tuning.columns.tolist():
    raise ValueError("Train tuning et Validation tuning ont des structures différentes.")

if X_train_tuning.columns.tolist() != X_test.columns.tolist():
    raise ValueError("La structure du Test final diffère de celle du tuning.")

tuning_datasets_df = pd.DataFrame(
    [
        {
            "jeu_de_donnees": "Apprentissage tuning",
            "source": "X_train_dev",
            "observations": len(X_train_tuning),
            "variables": X_train_tuning.shape[1],
            "utilisation": "Entraînement des configurations d'hyperparamètres",
        },
        {
            "jeu_de_donnees": "Validation tuning",
            "source": "X_validation_dev",
            "observations": len(X_validation_tuning),
            "variables": X_validation_tuning.shape[1],
            "utilisation": "Validation externe des meilleures configurations",
        },
        {
            "jeu_de_donnees": "Test final",
            "source": "X_test",
            "observations": len(X_test),
            "variables": X_test.shape[1],
            "utilisation": "Évaluation finale uniquement — exclu du tuning",
        },
    ]
)

print("\n" + "=" * 72)
print("RÉPARTITION DES DONNÉES POUR L'OPTIMISATION")
print("=" * 72)
display(tuning_datasets_df)

print("\n✅ Aucun nouveau découpage Train / Validation n'a été effectué.")
print("✅ X_train_dev / y_train_dev alimente la recherche d'hyperparamètres.")
print("✅ X_validation_dev / y_validation_dev reste une validation externe.")
print("✅ X_test / y_test reste totalement exclu du tuning.")


PRÉPARATION DES DONNÉES POUR L'OPTIMISATION

RÉPARTITION DES DONNÉES POUR L'OPTIMISATION


,jeu_de_donnees,source,observations,variables,utilisation
0,Apprentissage tuning,X_train_dev,400000,35,Entraînement des configurations d'hyperparamètres
1,Validation tuning,X_validation_dev,100000,35,Validation externe des meilleures configurations
2,Test final,X_test,2151637,35,Évaluation finale uniquement — exclu du tuning



✅ Aucun nouveau découpage Train / Validation n'a été effectué.
✅ X_train_dev / y_train_dev alimente la recherche d'hyperparamètres.
✅ X_validation_dev / y_validation_dev reste une validation externe.
✅ X_test / y_test reste totalement exclu du tuning.


### 8.2 Définition des espaces d'hyperparamètres

Les espaces de recherche sont volontairement bornés afin de maîtriser le coût
computationnel.

Pour Random Forest, les paramètres explorés concernent notamment le nombre
d'arbres, la profondeur, les contraintes de séparation et `max_features`.

Pour HistGradientBoosting, ils concernent notamment le taux d'apprentissage,
le nombre d'itérations, le nombre de feuilles, la taille minimale des feuilles
et la régularisation L2.

Aucun entraînement n'est lancé dans cette cellule.


In [15]:
# =====================================================================
# 8.2 - Définition des espaces d'hyperparamètres
# =====================================================================

from math import prod


random_forest_param_distributions = {
    "n_estimators": [100, 200, 300],
    "max_depth": [15, 20, 25, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.7, 1.0],
}

hist_gradient_boosting_param_distributions = {
    "learning_rate": [0.03, 0.05, 0.1],
    "max_iter": [100, 200, 300],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [20, 50, 100],
    "l2_regularization": [0.0, 0.1, 1.0, 5.0],
}

rf_combinations = prod(
    len(values) for values in random_forest_param_distributions.values()
)
hgb_combinations = prod(
    len(values) for values in hist_gradient_boosting_param_distributions.values()
)

hyperparameter_spaces_df = pd.DataFrame(
    [
        {
            "modele": "Random Forest",
            "hyperparametres": len(random_forest_param_distributions),
            "combinaisons_theoriques": rf_combinations,
        },
        {
            "modele": "HistGradientBoosting",
            "hyperparametres": len(hist_gradient_boosting_param_distributions),
            "combinaisons_theoriques": hgb_combinations,
        },
    ]
)

print("=" * 72)
print("DÉFINITION DES ESPACES D'HYPERPARAMÈTRES")
print("=" * 72)
display(hyperparameter_spaces_df)

print("\nRandom Forest Regressor :")
for parameter, values in random_forest_param_distributions.items():
    print(f"  - {parameter:<20} : {len(values)} valeur(s) -> {values}")
print(f"  Nombre théorique de combinaisons : {rf_combinations:,}")

print("\nHistGradientBoosting Regressor :")
for parameter, values in hist_gradient_boosting_param_distributions.items():
    print(f"  - {parameter:<20} : {len(values)} valeur(s) -> {values}")
print(f"  Nombre théorique de combinaisons : {hgb_combinations:,}")

print("\n✅ Espaces d'hyperparamètres définis.")
print("✅ Aucun entraînement n'a encore été lancé.")
print("✅ X_test / y_test reste exclu du tuning.")


DÉFINITION DES ESPACES D'HYPERPARAMÈTRES


,modele,hyperparametres,combinaisons_theoriques
0,Random Forest,5,324
1,HistGradientBoosting,5,324



Random Forest Regressor :
  - n_estimators         : 3 valeur(s) -> [100, 200, 300]
  - max_depth            : 4 valeur(s) -> [15, 20, 25, None]
  - min_samples_split    : 3 valeur(s) -> [2, 5, 10]
  - min_samples_leaf     : 3 valeur(s) -> [1, 2, 5]
  - max_features         : 3 valeur(s) -> ['sqrt', 0.7, 1.0]
  Nombre théorique de combinaisons : 324

HistGradientBoosting Regressor :
  - learning_rate        : 3 valeur(s) -> [0.03, 0.05, 0.1]
  - max_iter             : 3 valeur(s) -> [100, 200, 300]
  - max_leaf_nodes       : 3 valeur(s) -> [15, 31, 63]
  - min_samples_leaf     : 3 valeur(s) -> [20, 50, 100]
  - l2_regularization    : 4 valeur(s) -> [0.0, 0.1, 1.0, 5.0]
  Nombre théorique de combinaisons : 324

✅ Espaces d'hyperparamètres définis.
✅ Aucun entraînement n'a encore été lancé.
✅ X_test / y_test reste exclu du tuning.


### 8.3 Définition de la stratégie d'optimisation

La recherche repose sur `RandomizedSearchCV`, plus adaptée qu'une recherche
exhaustive sur les espaces définis.

La **MAE** est la métrique d'optimisation, via le scoring
`neg_mean_absolute_error`.

La recherche utilise une validation croisée interne à **3 folds** exclusivement
sur `X_train_tuning / y_train_tuning`.

`X_validation_tuning / y_validation_tuning` reste indépendant de
`RandomizedSearchCV` et sert ensuite à valider les meilleures configurations.

Le Test final reste exclu de toute cette phase.


In [16]:
# =====================================================================
# 8.3 - Définition de la stratégie d'optimisation
# =====================================================================

CV_FOLDS = 3
SCORING = "neg_mean_absolute_error"
TUNING_RANDOM_STATE = RANDOM_STATE

RF_N_ITER = 8
HGB_N_ITER = 20

rf_total_fits = RF_N_ITER * CV_FOLDS
hgb_total_fits = HGB_N_ITER * CV_FOLDS

if RF_N_ITER > rf_combinations:
    raise ValueError("RF_N_ITER dépasse l'espace Random Forest disponible.")
if HGB_N_ITER > hgb_combinations:
    raise ValueError("HGB_N_ITER dépasse l'espace HistGradientBoosting disponible.")
if CV_FOLDS < 2:
    raise ValueError("La validation croisée nécessite au minimum 2 folds.")

tuning_strategy_df = pd.DataFrame(
    [
        {
            "modele": "Random Forest",
            "espace_total": rf_combinations,
            "configurations_testees": RF_N_ITER,
            "folds_cv": CV_FOLDS,
            "entrainements_cv": rf_total_fits,
            "metrique_selection": "MAE",
        },
        {
            "modele": "HistGradientBoosting",
            "espace_total": hgb_combinations,
            "configurations_testees": HGB_N_ITER,
            "folds_cv": CV_FOLDS,
            "entrainements_cv": hgb_total_fits,
            "metrique_selection": "MAE",
        },
    ]
)

print("=" * 72)
print("DÉFINITION DE LA STRATÉGIE D'OPTIMISATION")
print("=" * 72)
display(tuning_strategy_df)

print(
    f"\nRandom Forest         : {RF_N_ITER} configurations × "
    f"{CV_FOLDS} folds = {rf_total_fits} entraînements CV"
)
print(
    f"HistGradientBoosting  : {HGB_N_ITER} configurations × "
    f"{CV_FOLDS} folds = {hgb_total_fits} entraînements CV"
)
print(f"\nMétrique principale   : MAE")
print(f"Scoring Scikit-learn  : {SCORING}")
print(f"Random state           : {TUNING_RANDOM_STATE}")

print(
    f"\nTrain tuning           : {len(X_train_tuning):,} observations "
    "-> utilisé par RandomizedSearchCV"
)
print(
    f"Validation tuning      : {len(X_validation_tuning):,} observations "
    "-> validation externe"
)
print(
    f"Test final             : {len(X_test):,} observations "
    "-> strictement exclu du tuning"
)

print("\n✅ Stratégie de tuning définie.")
print("✅ Aucun entraînement n'a encore été lancé.")


DÉFINITION DE LA STRATÉGIE D'OPTIMISATION


,modele,espace_total,configurations_testees,folds_cv,entrainements_cv,metrique_selection
0,Random Forest,324,8,3,24,MAE
1,HistGradientBoosting,324,20,3,60,MAE



Random Forest         : 8 configurations × 3 folds = 24 entraînements CV
HistGradientBoosting  : 20 configurations × 3 folds = 60 entraînements CV

Métrique principale   : MAE
Scoring Scikit-learn  : neg_mean_absolute_error
Random state           : 42

Train tuning           : 400,000 observations -> utilisé par RandomizedSearchCV
Validation tuning      : 100,000 observations -> validation externe
Test final             : 2,151,637 observations -> strictement exclu du tuning

✅ Stratégie de tuning définie.
✅ Aucun entraînement n'a encore été lancé.


### 8.4 Optimisation du Random Forest Regressor

Cette étape recherche une meilleure configuration de
`RandomForestRegressor` avec `RandomizedSearchCV`.

La recherche utilise uniquement `X_train_tuning / y_train_tuning`, avec la
stratégie définie à l'étape 8.3.

Pour limiter le risque de saturation mémoire, `RandomizedSearchCV` exécute les
configurations séquentiellement (`n_jobs=1`) tandis qu'un Random Forest peut
utiliser les cœurs disponibles (`n_jobs=-1`).

Le Test final n'est pas utilisé.


In [17]:
# =====================================================================
# 8.4 - Optimisation contrôlée du Random Forest
# =====================================================================

import time

from sklearn.model_selection import RandomizedSearchCV


random_forest_tuning_model = RandomForestRegressor(
    random_state=TUNING_RANDOM_STATE,
    n_jobs=-1,
)

random_forest_search = RandomizedSearchCV(
    estimator=random_forest_tuning_model,
    param_distributions=random_forest_param_distributions,
    n_iter=RF_N_ITER,
    scoring=SCORING,
    cv=CV_FOLDS,
    random_state=TUNING_RANDOM_STATE,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
    refit=True,
)

print("=" * 72)
print("OPTIMISATION RANDOM FOREST")
print("=" * 72)
print(f"\nObservations tuning : {len(X_train_tuning):,}")
print(f"Variables            : {X_train_tuning.shape[1]}")
print(f"Configurations       : {RF_N_ITER}")
print(f"Folds CV             : {CV_FOLDS}")
print(f"Entraînements prévus : {rf_total_fits}")
print("Mode de recherche    : séquentiel au niveau RandomizedSearchCV")

rf_tuning_start_time = time.perf_counter()
random_forest_search.fit(X_train_tuning, y_train_tuning)
rf_tuning_time_seconds = time.perf_counter() - rf_tuning_start_time

rf_best_params = random_forest_search.best_params_.copy()
rf_best_cv_mae = float(-random_forest_search.best_score_)
rf_best_model = random_forest_search.best_estimator_

rf_cv_results_df = pd.DataFrame(random_forest_search.cv_results_)
rf_cv_results_df["mae_train_cv"] = -rf_cv_results_df["mean_train_score"]
rf_cv_results_df["mae_validation_cv"] = -rf_cv_results_df["mean_test_score"]
rf_cv_results_df["std_mae_validation_cv"] = rf_cv_results_df["std_test_score"]

rf_tuning_results_df = (
    rf_cv_results_df[
        [
            "rank_test_score",
            "mae_train_cv",
            "mae_validation_cv",
            "std_mae_validation_cv",
            "mean_fit_time",
            "params",
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

rf_tuning_results_display_df = rf_tuning_results_df.copy()
for column in [
    "mae_train_cv",
    "mae_validation_cv",
    "std_mae_validation_cv",
    "mean_fit_time",
]:
    rf_tuning_results_display_df[column] = rf_tuning_results_display_df[column].round(4)

print("\n" + "=" * 72)
print("RÉSULTATS DU TUNING RANDOM FOREST")
print("=" * 72)
print(f"\nMeilleure MAE CV : {rf_best_cv_mae:.4f} g CO₂/km")
print(f"Temps total      : {rf_tuning_time_seconds:.2f} secondes")
print("\nMeilleurs hyperparamètres :")
for parameter, value in rf_best_params.items():
    print(f"  - {parameter}: {value}")

display(rf_tuning_results_display_df)

print("\n✅ Optimisation Random Forest terminée.")
print("✅ Le Test final n'a pas été utilisé.")


OPTIMISATION RANDOM FOREST

Observations tuning : 400,000
Variables            : 35
Configurations       : 8
Folds CV             : 3
Entraînements prévus : 24
Mode de recherche    : séquentiel au niveau RandomizedSearchCV
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=5, min_samples_split=10, n_estimators=100; total time= 1.1min
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=5, min_samples_split=10, n_estimators=100; total time= 1.1min
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=5, min_samples_split=10, n_estimators=100; total time= 1.3min
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time= 1.4min
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time= 1.3min
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time= 1.3min


,rank_test_score,mae_train_cv,mae_validation_cv,std_mae_validation_cv,mean_fit_time,params
0,1,0.2801,0.4974,0.0087,112.2440,"{'n_estimators': 100, 'min_samples_split': 5, ..."
1,2,0.3569,0.5222,0.0076,157.4442,"{'n_estimators': 200, 'min_samples_split': 10,..."
2,3,0.3398,0.5371,0.0089,77.7125,"{'n_estimators': 100, 'min_samples_split': 2, ..."
3,4,0.3379,0.5485,0.0097,348.5791,"{'n_estimators': 300, 'min_samples_split': 2, ..."
4,5,0.5381,0.6496,0.0094,67.3167,"{'n_estimators': 100, 'min_samples_split': 10,..."
5,6,0.5381,0.6496,0.0094,76.0172,"{'n_estimators': 100, 'min_samples_split': 2, ..."
6,7,1.6214,1.7141,0.0858,35.5343,"{'n_estimators': 100, 'min_samples_split': 2, ..."
7,8,1.9950,2.0816,0.0499,23.7223,"{'n_estimators': 100, 'min_samples_split': 2, ..."



✅ Optimisation Random Forest terminée.
✅ Le Test final n'a pas été utilisé.


### 8.5 Validation du Random Forest optimisé

La meilleure configuration issue de la validation croisée interne est évaluée
sur `X_validation_tuning / y_validation_tuning`, qui n'a pas participé à la
recherche des hyperparamètres.

Cette validation externe mesure MAE, RMSE et R² et compare la MAE observée à la
MAE moyenne de validation croisée.

Le Test final reste exclu.


In [18]:
# =====================================================================
# 8.5 - Validation externe du Random Forest optimisé
# =====================================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


y_rf_validation_pred = rf_best_model.predict(X_validation_tuning)

rf_validation_mae = float(
    mean_absolute_error(y_validation_tuning, y_rf_validation_pred)
)
rf_validation_rmse = float(
    np.sqrt(mean_squared_error(y_validation_tuning, y_rf_validation_pred))
)
rf_validation_r2 = float(
    r2_score(y_validation_tuning, y_rf_validation_pred)
)

rf_cv_validation_gap = rf_validation_mae - rf_best_cv_mae
rf_cv_validation_gap_abs = abs(rf_cv_validation_gap)
rf_cv_validation_gap_percent = (
    rf_cv_validation_gap_abs / rf_best_cv_mae * 100
    if rf_best_cv_mae != 0
    else np.nan
)

rf_optimized_validation_df = pd.DataFrame(
    [
        {
            "modele": "Random Forest optimisé",
            "jeu_evaluation": "Validation croisée interne",
            "observations": len(X_train_tuning),
            "MAE_g_co2_km": rf_best_cv_mae,
            "RMSE_g_co2_km": np.nan,
            "R2": np.nan,
        },
        {
            "modele": "Random Forest optimisé",
            "jeu_evaluation": "Validation externe",
            "observations": len(X_validation_tuning),
            "MAE_g_co2_km": rf_validation_mae,
            "RMSE_g_co2_km": rf_validation_rmse,
            "R2": rf_validation_r2,
        },
    ]
)

print("=" * 72)
print("VALIDATION DU RANDOM FOREST OPTIMISÉ")
print("=" * 72)
display(rf_optimized_validation_df.round(4))

print(
    f"\nMAE CV interne        : {rf_best_cv_mae:.4f} g CO₂/km"
    f"\nMAE Validation externe: {rf_validation_mae:.4f} g CO₂/km"
    f"\nRMSE Validation       : {rf_validation_rmse:.4f} g CO₂/km"
    f"\nR² Validation         : {rf_validation_r2:.4f}"
    f"\nÉcart MAE             : {rf_cv_validation_gap:+.4f} g CO₂/km"
    f"\nAmplitude relative    : {rf_cv_validation_gap_percent:.2f} %"
)

if rf_cv_validation_gap < 0:
    print("\n✅ La validation externe est meilleure que la moyenne CV.")
elif rf_cv_validation_gap_percent <= 5:
    print("\n✅ La validation externe est très proche de la moyenne CV.")
elif rf_cv_validation_gap_percent <= 10:
    print("\n⚠️ Dégradation modérée entre CV et validation externe.")
else:
    print("\n⚠️ Dégradation importante entre CV et validation externe.")

print("✅ X_test / y_test n'a pas été utilisé.")


VALIDATION DU RANDOM FOREST OPTIMISÉ


,modele,jeu_evaluation,observations,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Random Forest optimisé,Validation croisée interne,400000,0.4974,NaN,NaN
1,Random Forest optimisé,Validation externe,100000,0.4365,2.8801,0.9975



MAE CV interne        : 0.4974 g CO₂/km
MAE Validation externe: 0.4365 g CO₂/km
RMSE Validation       : 2.8801 g CO₂/km
R² Validation         : 0.9975
Écart MAE             : -0.0609 g CO₂/km
Amplitude relative    : 12.24 %

✅ La validation externe est meilleure que la moyenne CV.
✅ X_test / y_test n'a pas été utilisé.


### 8.6 Optimisation du HistGradientBoosting Regressor

Cette étape applique la même stratégie de recherche à
`HistGradientBoostingRegressor`.

La recherche est réalisée exclusivement sur `X_train_tuning / y_train_tuning`.
Le Test final reste exclu.


In [19]:
# =====================================================================
# 8.6 - Optimisation du HistGradientBoosting
# =====================================================================

hist_gradient_boosting_tuning_model = HistGradientBoostingRegressor(
    random_state=TUNING_RANDOM_STATE,
)

hist_gradient_boosting_search = RandomizedSearchCV(
    estimator=hist_gradient_boosting_tuning_model,
    param_distributions=hist_gradient_boosting_param_distributions,
    n_iter=HGB_N_ITER,
    scoring=SCORING,
    cv=CV_FOLDS,
    random_state=TUNING_RANDOM_STATE,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
    refit=True,
)

print("=" * 72)
print("OPTIMISATION HISTGRADIENTBOOSTING")
print("=" * 72)
print(f"\nObservations tuning : {len(X_train_tuning):,}")
print(f"Variables            : {X_train_tuning.shape[1]}")
print(f"Configurations       : {HGB_N_ITER}")
print(f"Folds CV             : {CV_FOLDS}")
print(f"Entraînements prévus : {hgb_total_fits}")

hgb_tuning_start_time = time.perf_counter()
hist_gradient_boosting_search.fit(X_train_tuning, y_train_tuning)
hgb_tuning_time_seconds = time.perf_counter() - hgb_tuning_start_time

hgb_best_params = hist_gradient_boosting_search.best_params_.copy()
hgb_best_cv_mae = float(-hist_gradient_boosting_search.best_score_)
hgb_best_model = hist_gradient_boosting_search.best_estimator_

hgb_cv_results_df = pd.DataFrame(hist_gradient_boosting_search.cv_results_)
hgb_cv_results_df["mae_train_cv"] = -hgb_cv_results_df["mean_train_score"]
hgb_cv_results_df["mae_validation_cv"] = -hgb_cv_results_df["mean_test_score"]
hgb_cv_results_df["std_mae_validation_cv"] = hgb_cv_results_df["std_test_score"]

hgb_tuning_results_df = (
    hgb_cv_results_df[
        [
            "rank_test_score",
            "mae_train_cv",
            "mae_validation_cv",
            "std_mae_validation_cv",
            "mean_fit_time",
            "params",
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

hgb_tuning_results_display_df = hgb_tuning_results_df.copy()
for column in [
    "mae_train_cv",
    "mae_validation_cv",
    "std_mae_validation_cv",
    "mean_fit_time",
]:
    hgb_tuning_results_display_df[column] = hgb_tuning_results_display_df[column].round(4)

print("\n" + "=" * 72)
print("RÉSULTATS DU TUNING HISTGRADIENTBOOSTING")
print("=" * 72)
print(f"\nMeilleure MAE CV : {hgb_best_cv_mae:.4f} g CO₂/km")
print(f"Temps total      : {hgb_tuning_time_seconds:.2f} secondes")
print("\nMeilleurs hyperparamètres :")
for parameter, value in hgb_best_params.items():
    print(f"  - {parameter}: {value}")

display(hgb_tuning_results_display_df)

print("\n✅ Optimisation HistGradientBoosting terminée.")
print("✅ Le Test final n'a pas été utilisé.")


OPTIMISATION HISTGRADIENTBOOSTING

Observations tuning : 400,000
Variables            : 35
Configurations       : 20
Folds CV             : 3
Entraînements prévus : 60
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=300, max_leaf_nodes=63, min_samples_leaf=20; total time=  37.6s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=300, max_leaf_nodes=63, min_samples_leaf=20; total time=  41.7s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=300, max_leaf_nodes=63, min_samples_leaf=20; total time=  41.6s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=100, max_leaf_nodes=15, min_samples_leaf=20; total time=  13.2s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=100, max_leaf_nodes=15, min_samples_leaf=20; total time=  10.9s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=100, max_leaf_nodes=15, min_samples_leaf=20; total time=  11.4s
[CV] END l2_regulariz

,rank_test_score,mae_train_cv,mae_validation_cv,std_mae_validation_cv,mean_fit_time,params
0,1,1.2122,1.2747,0.0122,35.2109,"{'min_samples_leaf': 20, 'max_leaf_nodes': 63,..."
1,2,1.3243,1.3676,0.0136,29.4179,"{'min_samples_leaf': 100, 'max_leaf_nodes': 63..."
2,3,1.3697,1.4120,0.0105,19.2408,"{'min_samples_leaf': 50, 'max_leaf_nodes': 63,..."
3,4,1.3949,1.4355,0.0077,32.3124,"{'min_samples_leaf': 50, 'max_leaf_nodes': 63,..."
4,5,1.3886,1.4378,0.0064,31.4380,"{'min_samples_leaf': 20, 'max_leaf_nodes': 63,..."
5,6,1.6087,1.6375,0.0113,23.4976,"{'min_samples_leaf': 100, 'max_leaf_nodes': 63..."
6,7,1.7319,1.7618,0.0134,14.2201,"{'min_samples_leaf': 50, 'max_leaf_nodes': 63,..."
7,8,1.7566,1.7805,0.0119,13.5158,"{'min_samples_leaf': 100, 'max_leaf_nodes': 63..."
8,9,1.7818,1.8178,0.0162,10.2725,"{'min_samples_leaf': 20, 'max_leaf_nodes': 31,..."
9,10,1.8052,1.8278,0.0188,8.4179,"{'min_samples_leaf': 100, 'max_leaf_nodes': 31..."



✅ Optimisation HistGradientBoosting terminée.
✅ Le Test final n'a pas été utilisé.


### 8.7 Validation du HistGradientBoosting optimisé

La meilleure configuration HistGradientBoosting est évaluée sur le même jeu de
validation externe que Random Forest.

Cette étape permet une comparaison équitable des deux modèles optimisés sans
utiliser le Test final.


In [20]:
# =====================================================================
# 8.7 - Validation externe du HistGradientBoosting optimisé
# =====================================================================

y_hgb_validation_pred = hgb_best_model.predict(X_validation_tuning)

hgb_validation_mae = float(
    mean_absolute_error(y_validation_tuning, y_hgb_validation_pred)
)
hgb_validation_rmse = float(
    np.sqrt(mean_squared_error(y_validation_tuning, y_hgb_validation_pred))
)
hgb_validation_r2 = float(
    r2_score(y_validation_tuning, y_hgb_validation_pred)
)

hgb_cv_validation_gap = hgb_validation_mae - hgb_best_cv_mae
hgb_cv_validation_gap_abs = abs(hgb_cv_validation_gap)
hgb_cv_validation_gap_percent = (
    hgb_cv_validation_gap_abs / hgb_best_cv_mae * 100
    if hgb_best_cv_mae != 0
    else np.nan
)

hgb_optimized_validation_df = pd.DataFrame(
    [
        {
            "modele": "HistGradientBoosting optimisé",
            "jeu_evaluation": "Validation croisée interne",
            "observations": len(X_train_tuning),
            "MAE_g_co2_km": hgb_best_cv_mae,
            "RMSE_g_co2_km": np.nan,
            "R2": np.nan,
        },
        {
            "modele": "HistGradientBoosting optimisé",
            "jeu_evaluation": "Validation externe",
            "observations": len(X_validation_tuning),
            "MAE_g_co2_km": hgb_validation_mae,
            "RMSE_g_co2_km": hgb_validation_rmse,
            "R2": hgb_validation_r2,
        },
    ]
)

print("=" * 72)
print("VALIDATION DU HISTGRADIENTBOOSTING OPTIMISÉ")
print("=" * 72)
display(hgb_optimized_validation_df.round(4))

print(
    f"\nMAE CV interne        : {hgb_best_cv_mae:.4f} g CO₂/km"
    f"\nMAE Validation externe: {hgb_validation_mae:.4f} g CO₂/km"
    f"\nRMSE Validation       : {hgb_validation_rmse:.4f} g CO₂/km"
    f"\nR² Validation         : {hgb_validation_r2:.4f}"
    f"\nÉcart MAE             : {hgb_cv_validation_gap:+.4f} g CO₂/km"
    f"\nAmplitude relative    : {hgb_cv_validation_gap_percent:.2f} %"
)

if hgb_cv_validation_gap < 0:
    print("\n✅ La validation externe est meilleure que la moyenne CV.")
elif hgb_cv_validation_gap_percent <= 5:
    print("\n✅ La validation externe est très proche de la moyenne CV.")
elif hgb_cv_validation_gap_percent <= 10:
    print("\n⚠️ Dégradation modérée entre CV et validation externe.")
else:
    print("\n⚠️ Dégradation importante entre CV et validation externe.")

print("✅ X_test / y_test n'a pas été utilisé.")


VALIDATION DU HISTGRADIENTBOOSTING OPTIMISÉ


,modele,jeu_evaluation,observations,MAE_g_co2_km,RMSE_g_co2_km,R2
0,HistGradientBoosting optimisé,Validation croisée interne,400000,1.2747,NaN,NaN
1,HistGradientBoosting optimisé,Validation externe,100000,1.2705,3.7809,0.9957



MAE CV interne        : 1.2747 g CO₂/km
MAE Validation externe: 1.2705 g CO₂/km
RMSE Validation       : 3.7809 g CO₂/km
R² Validation         : 0.9957
Écart MAE             : -0.0042 g CO₂/km
Amplitude relative    : 0.33 %

✅ La validation externe est meilleure que la moyenne CV.
✅ X_test / y_test n'a pas été utilisé.


### 8.8 Comparaison des modèles optimisés

Les meilleures configurations Random Forest et HistGradientBoosting sont
comparées sur exactement le même jeu Validation externe.

Le **RMSE de validation** constitue le critère principal de sélection du modèle
candidat final. La MAE, le R² et le coût du tuning restent des critères
complémentaires.

Le Test final n'est toujours pas utilisé.


In [21]:
# =====================================================================
# 8.8 - Comparaison des modèles optimisés
# =====================================================================

optimized_models_comparison_df = pd.DataFrame(
    [
        {
            "model": "Random Forest",
            "mae_validation": rf_validation_mae,
            "rmse_validation": rf_validation_rmse,
            "r2_validation": rf_validation_r2,
            "best_cv_mae": rf_best_cv_mae,
            "tuning_time_seconds": rf_tuning_time_seconds,
        },
        {
            "model": "HistGradientBoosting",
            "mae_validation": hgb_validation_mae,
            "rmse_validation": hgb_validation_rmse,
            "r2_validation": hgb_validation_r2,
            "best_cv_mae": hgb_best_cv_mae,
            "tuning_time_seconds": hgb_tuning_time_seconds,
        },
    ]
).sort_values("rmse_validation").reset_index(drop=True)

optimized_models_display_df = optimized_models_comparison_df.copy()
numeric_columns = optimized_models_display_df.select_dtypes(include="number").columns
optimized_models_display_df[numeric_columns] = optimized_models_display_df[numeric_columns].round(4)

print("=" * 72)
print("COMPARAISON DES MODÈLES OPTIMISÉS")
print("=" * 72)
print("\nClassement principal : RMSE Validation externe croissant.\n")
display(optimized_models_display_df)

final_candidate_name = optimized_models_comparison_df.iloc[0]["model"]

if final_candidate_name == "Random Forest":
    final_candidate_model = rf_best_model
    final_candidate_params = rf_best_params.copy()
    final_candidate_validation_metrics = {
        "mae": rf_validation_mae,
        "rmse": rf_validation_rmse,
        "r2": rf_validation_r2,
    }
else:
    final_candidate_model = hgb_best_model
    final_candidate_params = hgb_best_params.copy()
    final_candidate_validation_metrics = {
        "mae": hgb_validation_mae,
        "rmse": hgb_validation_rmse,
        "r2": hgb_validation_r2,
    }

print(
    f"\n✅ Modèle candidat final sur Validation externe : "
    f"{final_candidate_name}"
)
print(
    f"   - MAE  : {final_candidate_validation_metrics['mae']:.4f} g CO₂/km"
    f"\n   - RMSE : {final_candidate_validation_metrics['rmse']:.4f} g CO₂/km"
    f"\n   - R²   : {final_candidate_validation_metrics['r2']:.4f}"
)
print("✅ X_test / y_test n'a pas encore été utilisé.")


COMPARAISON DES MODÈLES OPTIMISÉS

Classement principal : RMSE Validation externe croissant.



,model,mae_validation,rmse_validation,r2_validation,best_cv_mae,tuning_time_seconds
0,Random Forest,0.4365,2.8801,0.9975,0.4974,3116.6638
1,HistGradientBoosting,1.2705,3.7809,0.9957,1.2747,1482.0436



✅ Modèle candidat final sur Validation externe : Random Forest
   - MAE  : 0.4365 g CO₂/km
   - RMSE : 2.8801 g CO₂/km
   - R²   : 0.9975
✅ X_test / y_test n'a pas encore été utilisé.


### 8.9 Évaluation finale du modèle candidat sur le Test final

Après sélection du modèle candidat sur Validation externe, la configuration
retenue est réentraînée sur l'ensemble du sous-échantillon de développement
(`X_development / y_development`).

Le Test final est ensuite utilisé **une seule fois** pour mesurer MAE, RMSE et
R².

Aucun hyperparamètre n'est modifié à partir des résultats du Test final.


In [22]:
# =====================================================================
# 8.9 - Évaluation finale sur le Test final
# =====================================================================

from sklearn.base import clone


print("=" * 72)
print("ÉVALUATION FINALE DU MODÈLE CANDIDAT")
print("=" * 72)

print(f"\nModèle candidat : {final_candidate_name}")
print(f"Développement utilisé pour le réentraînement : {len(X_development):,} observations")
print(f"Test final réservé : {len(X_test):,} observations")

# Le clone reprend la configuration sélectionnée mais aucun état appris.
final_candidate_development_model = clone(final_candidate_model)

refit_start_time = time.perf_counter()
final_candidate_development_model.fit(X_development, y_development)
final_candidate_refit_time_seconds = time.perf_counter() - refit_start_time

y_test_pred = final_candidate_development_model.predict(X_test)

final_test_mae = float(mean_absolute_error(y_test, y_test_pred))
final_test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_pred)))
final_test_r2 = float(r2_score(y_test, y_test_pred))

final_test_evaluation_df = pd.DataFrame(
    [
        {
            "model": final_candidate_name,
            "dataset": "Validation externe",
            "observations": len(X_validation_tuning),
            "MAE_g_co2_km": final_candidate_validation_metrics["mae"],
            "RMSE_g_co2_km": final_candidate_validation_metrics["rmse"],
            "R2": final_candidate_validation_metrics["r2"],
        },
        {
            "model": final_candidate_name,
            "dataset": "Test final",
            "observations": len(X_test),
            "MAE_g_co2_km": final_test_mae,
            "RMSE_g_co2_km": final_test_rmse,
            "R2": final_test_r2,
        },
    ]
)

print("\n" + "=" * 72)
print("RÉSULTATS DE L'ÉVALUATION FINALE")
print("=" * 72)
display(final_test_evaluation_df.round(4))

print(
    f"\nMAE Test final  : {final_test_mae:.4f} g CO₂/km"
    f"\nRMSE Test final : {final_test_rmse:.4f} g CO₂/km"
    f"\nR² Test final   : {final_test_r2:.4f}"
    f"\nTemps réentraînement développement : {final_candidate_refit_time_seconds:.2f} s"
)

print("\n✅ Le Test final a été utilisé uniquement après la sélection du candidat.")
print("✅ Aucun hyperparamètre n'est ajusté à partir du Test final.")


ÉVALUATION FINALE DU MODÈLE CANDIDAT

Modèle candidat : Random Forest
Développement utilisé pour le réentraînement : 500,000 observations
Test final réservé : 2,151,637 observations

RÉSULTATS DE L'ÉVALUATION FINALE


,model,dataset,observations,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Random Forest,Validation externe,100000,0.4365,2.8801,0.9975
1,Random Forest,Test final,2151637,0.4134,2.8643,0.9976



MAE Test final  : 0.4134 g CO₂/km
RMSE Test final : 2.8643 g CO₂/km
R² Test final   : 0.9976
Temps réentraînement développement : 236.92 s

✅ Le Test final a été utilisé uniquement après la sélection du candidat.
✅ Aucun hyperparamètre n'est ajusté à partir du Test final.


### 8.10 Préparation du réentraînement FULL du modèle final

La configuration finale est maintenant connue. Cette étape prépare une
instance **non entraînée** du modèle sélectionné avec les hyperparamètres issus
du tuning.

L'entraînement FULL doit être exécuté ensuite dans l'environnement disposant
des ressources adaptées, sur l'intégralité de `X_train / y_train`.

Aucun nouvel ajustement d'hyperparamètres n'est réalisé.


In [23]:
# =====================================================================
# 8.10 - Préparation du modèle final pour le réentraînement FULL
# =====================================================================

if final_candidate_name == "Random Forest":
    final_model_params = final_candidate_params.copy()
    final_model_params.update(
        {
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }
    )
    final_model_for_full = RandomForestRegressor(**final_model_params)
    FINAL_MODEL_NAME = "random_forest_regressor"
    FINAL_MODEL_CLASS = "RandomForestRegressor"

elif final_candidate_name == "HistGradientBoosting":
    final_model_params = final_candidate_params.copy()
    final_model_params.update({"random_state": RANDOM_STATE})
    final_model_for_full = HistGradientBoostingRegressor(**final_model_params)
    FINAL_MODEL_NAME = "hist_gradient_boosting_regressor"
    FINAL_MODEL_CLASS = "HistGradientBoostingRegressor"

else:
    raise ValueError(f"Modèle candidat non géré : {final_candidate_name}")

print("=" * 72)
print("PRÉPARATION DU MODÈLE FINAL POUR TRAIN FULL")
print("=" * 72)
print(f"\nModèle final      : {FINAL_MODEL_CLASS}")
print(f"Train FULL        : {len(X_train):,} observations")
print(f"Variables         : {X_train.shape[1]}")
print("\nHyperparamètres :")
for parameter, value in final_model_params.items():
    print(f"  - {parameter}: {value}")

print("\n✅ Instance finale préparée sans entraînement FULL.")
print("✅ Les hyperparamètres proviennent uniquement du tuning.")


PRÉPARATION DU MODÈLE FINAL POUR TRAIN FULL

Modèle final      : RandomForestRegressor
Train FULL        : 8,606,547 observations
Variables         : 35

Hyperparamètres :
  - n_estimators: 100
  - min_samples_split: 5
  - min_samples_leaf: 1
  - max_features: 1.0
  - max_depth: 25
  - random_state: 42
  - n_jobs: -1

✅ Instance finale préparée sans entraînement FULL.
✅ Les hyperparamètres proviennent uniquement du tuning.


### 8.11 Industrialisation du réentraînement final

Cette étape prépare les informations nécessaires au script d'entraînement
industrialisé.

Le modèle FULL doit être entraîné en dehors du notebook afin de :

- reproduire l'entraînement ;
- exploiter un environnement disposant de ressources suffisantes ;
- sauvegarder l'artefact du modèle ;
- intégrer l'entraînement au pipeline DVC / MLOps.

Dans l'architecture actuelle du projet, le stage DVC de régression et le script
d'industrialisation sont historiquement configurés pour Random Forest. Si la
sélection dynamique retient une autre famille de modèle, ces éléments devront
être adaptés avant l'exécution FULL.


In [24]:
# =====================================================================
# 8.11 - Préparation des informations d'industrialisation
# =====================================================================

trained_models_dir = project_root / "models" / "trained"
trained_models_dir.mkdir(parents=True, exist_ok=True)

final_training_config = {
    "model_name": FINAL_MODEL_NAME,
    "model_class": FINAL_MODEL_CLASS,
    "target": TARGET_COLUMN,
    "train_observations": int(len(X_train)),
    "train_features": int(X_train.shape[1]),
    "test_observations": int(len(X_test)),
    "test_features": int(X_test.shape[1]),
    "model_directory": str(trained_models_dir),
    "hyperparameters": final_model_params,
}

final_training_report_df = pd.DataFrame(
    [
        {"element": "Modèle final", "valeur": FINAL_MODEL_CLASS},
        {"element": "Cible", "valeur": TARGET_COLUMN},
        {"element": "Train FULL", "valeur": f"{len(X_train):,} observations"},
        {"element": "Test final", "valeur": f"{len(X_test):,} observations"},
        {"element": "Nombre de variables", "valeur": X_train.shape[1]},
    ]
)

print("=" * 72)
print("CONFIGURATION D'INDUSTRIALISATION")
print("=" * 72)
display(final_training_report_df)

print(f"\nRépertoire de destination : {trained_models_dir}")
print("\nHyperparamètres à industrialiser :")
for parameter, value in final_model_params.items():
    print(f"  - {parameter}: {value}")

if X_train.shape[1] != X_test.shape[1]:
    raise ValueError("Le nombre de variables Train et Test est différent.")

if not trained_models_dir.is_dir():
    raise FileNotFoundError("Le répertoire models/trained n'a pas pu être créé.")

if FINAL_MODEL_CLASS != "RandomForestRegressor":
    print(
        "\n⚠️ Le modèle sélectionné n'est pas RandomForestRegressor. "
        "Le stage DVC et le script d'entraînement de régression actuels "
        "doivent être vérifiés avant l'entraînement FULL."
    )
else:
    print(
        "\n✅ La famille sélectionnée est compatible avec le stage "
        "Random Forest actuellement prévu dans le pipeline."
    )

print("✅ Aucun entraînement FULL n'a encore été lancé.")


CONFIGURATION D'INDUSTRIALISATION


,element,valeur
0,Modèle final,RandomForestRegressor
1,Cible,co2_wltp_g_km
2,Train FULL,"8,606,547 observations"
3,Test final,"2,151,637 observations"
4,Nombre de variables,35



Répertoire de destination : /home/jmbandong/projects/ml-projects/vehicle-emissions-prediction-mlops/models/trained

Hyperparamètres à industrialiser :
  - n_estimators: 100
  - min_samples_split: 5
  - min_samples_leaf: 1
  - max_features: 1.0
  - max_depth: 25
  - random_state: 42
  - n_jobs: -1

✅ La famille sélectionnée est compatible avec le stage Random Forest actuellement prévu dans le pipeline.
✅ Aucun entraînement FULL n'a encore été lancé.


### 8.12 Sauvegarde des résultats et métadonnées de l'expérimentation

Les résultats nécessaires à la traçabilité sont sauvegardés dans :

- `reports/tables/` pour les tableaux ;
- `models/metadata/` pour les métadonnées JSON.

Les valeurs sont générées à partir de l'exécution courante. Aucun résultat
expérimental n'est inscrit en dur.


In [25]:
# =====================================================================
# 8.12 - Sauvegarde des résultats et métadonnées
# =====================================================================

import json


reports_tables_dir = project_root / "reports" / "tables"
models_metadata_dir = project_root / "models" / "metadata"
reports_tables_dir.mkdir(parents=True, exist_ok=True)
models_metadata_dir.mkdir(parents=True, exist_ok=True)

regression_comparison_path = reports_tables_dir / "regression_model_comparison.csv"
rf_tuning_results_path = reports_tables_dir / "random_forest_tuning_results.csv"
rf_validation_path = reports_tables_dir / "random_forest_optimized_validation.csv"
hgb_tuning_results_path = reports_tables_dir / "hist_gradient_boosting_tuning_results.csv"
hgb_validation_path = reports_tables_dir / "hist_gradient_boosting_optimized_validation.csv"
optimized_models_comparison_path = reports_tables_dir / "optimized_models_comparison.csv"
final_test_evaluation_path = reports_tables_dir / "regression_final_test_evaluation.csv"

regression_comparison_df.to_csv(regression_comparison_path, index=False)
rf_tuning_results_df.to_csv(rf_tuning_results_path, index=False)
rf_optimized_validation_df.to_csv(rf_validation_path, index=False)
hgb_tuning_results_df.to_csv(hgb_tuning_results_path, index=False)
hgb_optimized_validation_df.to_csv(hgb_validation_path, index=False)
optimized_models_comparison_df.to_csv(optimized_models_comparison_path, index=False)
final_test_evaluation_df.to_csv(final_test_evaluation_path, index=False)


def to_python_scalar(value):
    """Convertit les scalaires NumPy en types JSON natifs."""
    return value.item() if hasattr(value, "item") else value


selection_metadata = {
    "target": TARGET_COLUMN,
    "excluded_features": ["manufacturer_make"],
    "feature_count": int(X_train.shape[1]),
    "features": X_train.columns.tolist(),
    "development": {
        "total_observations": int(len(X_development)),
        "train_observations": int(len(X_train_dev)),
        "validation_observations": int(len(X_validation_dev)),
        "random_state": int(RANDOM_STATE),
    },
    "tuning": {
        "train_observations": int(len(X_train_tuning)),
        "external_validation_observations": int(len(X_validation_tuning)),
        "cv_folds": int(CV_FOLDS),
        "scoring": SCORING,
        "random_state": int(TUNING_RANDOM_STATE),
    },
    "selected_model": {
        "display_name": final_candidate_name,
        "model_name": FINAL_MODEL_NAME,
        "model_class": FINAL_MODEL_CLASS,
        "best_params": {
            key: to_python_scalar(value)
            for key, value in final_candidate_params.items()
        },
    },
    "validation_metrics": {
        key: float(value)
        for key, value in final_candidate_validation_metrics.items()
    },
    "final_test": {
        "observations": int(len(X_test)),
        "mae": float(final_test_mae),
        "rmse": float(final_test_rmse),
        "r2": float(final_test_r2),
    },
}

selection_metadata_path = (
    models_metadata_dir / "regression_model_selection_metadata.json"
)

with open(selection_metadata_path, "w", encoding="utf-8") as file:
    json.dump(selection_metadata, file, indent=4, ensure_ascii=False)

# Métadonnées spécifiques au modèle sélectionné.
if FINAL_MODEL_CLASS == "RandomForestRegressor":
    selected_model_metadata_path = (
        models_metadata_dir / "random_forest_regression_metadata.json"
    )
else:
    selected_model_metadata_path = (
        models_metadata_dir / "hist_gradient_boosting_regression_metadata.json"
    )

with open(selected_model_metadata_path, "w", encoding="utf-8") as file:
    json.dump(selection_metadata, file, indent=4, ensure_ascii=False)

saved_artifacts = {
    "Comparaison modèles initiaux": regression_comparison_path,
    "Tuning Random Forest": rf_tuning_results_path,
    "Validation Random Forest": rf_validation_path,
    "Tuning HistGradientBoosting": hgb_tuning_results_path,
    "Validation HistGradientBoosting": hgb_validation_path,
    "Comparaison modèles optimisés": optimized_models_comparison_path,
    "Évaluation Test final": final_test_evaluation_path,
    "Métadonnées sélection": selection_metadata_path,
    "Métadonnées modèle final": selected_model_metadata_path,
}

saved_results_df = pd.DataFrame(
    [
        {
            "type": name,
            "fichier": path.name,
            "repertoire": str(path.parent.relative_to(project_root)),
        }
        for name, path in saved_artifacts.items()
    ]
)

print("=" * 72)
print("SAUVEGARDE DES RÉSULTATS ET MÉTADONNÉES")
print("=" * 72)
display(saved_results_df)

print("\n✅ Résultats expérimentaux sauvegardés.")
print("✅ Métadonnées de sélection sauvegardées.")
print("✅ Métadonnées spécifiques au modèle final sauvegardées.")
print("✅ Aucun modèle FULL n'est sauvegardé par ce notebook.")


SAUVEGARDE DES RÉSULTATS ET MÉTADONNÉES


,type,fichier,repertoire
0,Comparaison modèles initiaux,regression_model_comparison.csv,reports/tables
1,Tuning Random Forest,random_forest_tuning_results.csv,reports/tables
2,Validation Random Forest,random_forest_optimized_validation.csv,reports/tables
3,Tuning HistGradientBoosting,hist_gradient_boosting_tuning_results.csv,reports/tables
4,Validation HistGradientBoosting,hist_gradient_boosting_optimized_validation.csv,reports/tables
5,Comparaison modèles optimisés,optimized_models_comparison.csv,reports/tables
6,Évaluation Test final,regression_final_test_evaluation.csv,reports/tables
7,Métadonnées sélection,regression_model_selection_metadata.json,models/metadata
8,Métadonnées modèle final,random_forest_regression_metadata.json,models/metadata



✅ Résultats expérimentaux sauvegardés.
✅ Métadonnées de sélection sauvegardées.
✅ Métadonnées spécifiques au modèle final sauvegardées.
✅ Aucun modèle FULL n'est sauvegardé par ce notebook.


### 8.13 Validation des résultats sauvegardés

Cette dernière étape vérifie que les fichiers produits existent, sont relisibles
et contiennent les informations indispensables à la suite du pipeline.

Les contrôles portent notamment sur :

- l'existence et la non-vacuité des tableaux CSV ;
- la relecture des métadonnées JSON ;
- l'identité du nombre et de l'ordre des variables ;
- la présence des hyperparamètres et métriques finales ;
- la cohérence du modèle sélectionné.


In [26]:
# =====================================================================
# 8.13 - Validation des résultats sauvegardés
# =====================================================================

print("=" * 72)
print("VALIDATION DES RÉSULTATS SAUVEGARDÉS")
print("=" * 72)

checks = {}

# ---------------------------------------------------------------------
# 1. Existence de tous les artefacts
# ---------------------------------------------------------------------

for name, path in saved_artifacts.items():
    checks[f"Existe - {name}"] = path.is_file()


# ---------------------------------------------------------------------
# 2. Relecture des CSV
# ---------------------------------------------------------------------

csv_paths = [
    regression_comparison_path,
    rf_tuning_results_path,
    rf_validation_path,
    hgb_tuning_results_path,
    hgb_validation_path,
    optimized_models_comparison_path,
    final_test_evaluation_path,
]

for path in csv_paths:
    try:
        loaded_df = pd.read_csv(path)
        checks[f"CSV relisible - {path.name}"] = not loaded_df.empty
    except Exception:
        checks[f"CSV relisible - {path.name}"] = False


# ---------------------------------------------------------------------
# 3. Relecture et validation des métadonnées
# ---------------------------------------------------------------------

try:
    with open(selection_metadata_path, "r", encoding="utf-8") as file:
        loaded_metadata = json.load(file)

    checks["JSON sélection relisible"] = True
    checks["Cible conforme"] = loaded_metadata.get("target") == TARGET_COLUMN
    checks["Nombre de variables conforme"] = (
        loaded_metadata.get("feature_count") == X_train.shape[1]
    )
    checks["Ordre des variables conforme"] = (
        loaded_metadata.get("features") == X_train.columns.tolist()
    )
    checks["manufacturer_make exclue"] = (
        "manufacturer_make" in loaded_metadata.get("excluded_features", [])
        and all(
            "manufacturer_make" not in column.lower()
            for column in loaded_metadata.get("features", [])
        )
    )
    checks["Hyperparamètres présents"] = bool(
        loaded_metadata.get("selected_model", {}).get("best_params")
    )
    checks["Métriques Test final présentes"] = all(
        metric in loaded_metadata.get("final_test", {})
        for metric in ("mae", "rmse", "r2")
    )
    checks["Classe modèle conforme"] = (
        loaded_metadata.get("selected_model", {}).get("model_class")
        == FINAL_MODEL_CLASS
    )

except Exception:
    checks["JSON sélection relisible"] = False

try:
    with open(selected_model_metadata_path, "r", encoding="utf-8") as file:
        loaded_selected_model_metadata = json.load(file)
    checks["JSON modèle final relisible"] = (
        loaded_selected_model_metadata.get("selected_model", {}).get("model_class")
        == FINAL_MODEL_CLASS
    )
except Exception:
    checks["JSON modèle final relisible"] = False


# ---------------------------------------------------------------------
# 4. Rapport de validation
# ---------------------------------------------------------------------

validation_report_df = pd.DataFrame(
    [
        {
            "controle": name,
            "statut": "✅ Conforme" if result else "❌ Erreur",
        }
        for name, result in checks.items()
    ]
)

display(validation_report_df)

failed_checks = [name for name, result in checks.items() if not result]

if failed_checks:
    raise ValueError(
        "Validation des sauvegardes échouée : " + ", ".join(failed_checks)
    )

print("\n✅ Tous les fichiers attendus existent et sont relisibles.")
print("✅ Les métadonnées sont cohérentes avec les données utilisées.")
print("✅ L'ordre des variables du modèle est enregistré.")
print("✅ Les hyperparamètres et métriques finales sont disponibles.")
print("\nNotebook 04 terminé avec succès.")


VALIDATION DES RÉSULTATS SAUVEGARDÉS


,controle,statut
0,Existe - Comparaison modèles initiaux,✅ Conforme
1,Existe - Tuning Random Forest,✅ Conforme
2,Existe - Validation Random Forest,✅ Conforme
3,Existe - Tuning HistGradientBoosting,✅ Conforme
4,Existe - Validation HistGradientBoosting,✅ Conforme
5,Existe - Comparaison modèles optimisés,✅ Conforme
6,Existe - Évaluation Test final,✅ Conforme
7,Existe - Métadonnées sélection,✅ Conforme
8,Existe - Métadonnées modèle final,✅ Conforme
9,CSV relisible - regression_model_comparison.csv,✅ Conforme



✅ Tous les fichiers attendus existent et sont relisibles.
✅ Les métadonnées sont cohérentes avec les données utilisées.
✅ L'ordre des variables du modèle est enregistré.
✅ Les hyperparamètres et métriques finales sont disponibles.

Notebook 04 terminé avec succès.
